In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')

In [ ]:
uploaded = files.upload()
file_name = next(name for name in uploaded if name.endswith('.csv'))
df = pd.read_csv(file_name)

print('Loaded file:', file_name)
print('Dataset Shape:', df.shape)
display(df.head())

In [ ]:
print('Column Names')
print(df.columns.tolist())

print('Data Types')
print(df.dtypes)

print('Missing Values')
print(df.isnull().sum())

print('Duplicate Records:', df.duplicated().sum())

In [ ]:
# Columns selected for the 2026 Australian Grand Prix Practice 1 CSV.
# Position is not used because it is empty in this dataset.
features = ['LapNumber', 'Stint', 'TyreLife', 'Compound', 'SpeedI1', 'SpeedI2', 'SpeedST', 'AirTemp', 'TrackTemp', 'Humidity', 'WindSpeed']
target = 'LapTimeSeconds'

model_df = df[features + [target]].copy()

# A lap without a measured lap time cannot teach the model, so remove it.
model_df = model_df.dropna()
# Remove identical records so that a lap is not counted twice.
model_df = model_df.drop_duplicates()
# Keep only valid, positive lap times.
model_df = model_df[model_df[target] > 0]

print('Records after cleaning:', len(model_df))
display(model_df.head())

In [ ]:
# Histogram
plt.figure(figsize=(8, 4))
plt.hist(model_df[target], bins=25, edgecolor='black', color='skyblue')
plt.title('Distribution of Lap Times')
plt.xlabel('Lap Time (seconds)')
plt.ylabel('Number of Lap Records')
plt.show()

In [ ]:
# Count Plot
plt.figure(figsize=(7, 4))
sns.countplot(data=model_df, x='Compound', order=model_df['Compound'].value_counts().index)
plt.title('Number of Lap Records by Tyre Compound')
plt.xlabel('Tyre Compound')
plt.ylabel('Number of Lap Records')
plt.show()

# Bar Chart
average_time = model_df.groupby('Compound')[target].mean().reset_index()
plt.figure(figsize=(7, 4))
sns.barplot(data=average_time, x='Compound', y=target)
plt.title('Average Lap Time by Tyre Compound')
plt.xlabel('Tyre Compound')
plt.ylabel('Average Lap Time (seconds)')
plt.show()

In [ ]:
# Box Plot
plt.figure(figsize=(8, 4))
sns.boxplot(data=model_df, x='Compound', y=target)
plt.title('Lap Time Spread by Tyre Compound')
plt.xlabel('Tyre Compound')
plt.ylabel('Lap Time (seconds)')
plt.show()

# Scatter Plot
plt.figure(figsize=(8, 4))
sns.scatterplot(data=model_df, x='TyreLife', y=target, hue='Compound')
plt.title('Tyre Life versus Lap Time')
plt.xlabel('Tyre Life (laps)')
plt.ylabel('Lap Time (seconds)')
plt.show()

In [ ]:
# Correlation Heatmap
numeric_columns = ['LapNumber', 'Stint', 'TyreLife', 'SpeedI1', 'SpeedI2', 'SpeedST', 'AirTemp', 'TrackTemp', target]
plt.figure(figsize=(8, 5))
sns.heatmap(model_df[numeric_columns].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# Pair Plot
sns.pairplot(model_df[['LapNumber', 'TyreLife', 'SpeedST', 'TrackTemp', target]].sample(min(500, len(model_df)), random_state=42), corner=True)
plt.show()

In [ ]:
# Line Chart
lap_trend = model_df.groupby('LapNumber')[target].median().reset_index()
plt.figure(figsize=(10, 4))
sns.lineplot(data=lap_trend, x='LapNumber', y=target, marker='o')
plt.title('Median Lap Time by Lap Number')
plt.xlabel('Lap Number')
plt.ylabel('Median Lap Time (seconds)')
plt.show()

In [ ]:
X = model_df[features]
y = model_df[target]

# Train-test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Train-Test Split')
print('Training Records:', len(X_train))
print('Test Set Size:', len(X_test))

In [ ]:
# Convert the Compound column into numeric columns.
X_train = pd.get_dummies(X_train, columns=['Compound'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['Compound'], drop_first=True)

# Ensure training and test data have the same columns.
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Data Preparation Complete')
print('Training Data Shape:', X_train_scaled.shape)
print('Test Data Shape:', X_test_scaled.shape)

In [ ]:
# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
linear_pred = linear_model.predict(X_test_scaled)

print('Linear Regression Results')
print('MAE:', mean_absolute_error(y_test, linear_pred))
print('MSE:', mean_squared_error(y_test, linear_pred))
print('RMSE:', mean_squared_error(y_test, linear_pred) ** 0.5)
print('R2 Score:', r2_score(y_test, linear_pred))

In [ ]:
# Decision Tree Regressor
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train_scaled, y_train)
tree_pred = tree_model.predict(X_test_scaled)

print('Decision Tree Results')
print('MAE:', mean_absolute_error(y_test, tree_pred))
print('MSE:', mean_squared_error(y_test, tree_pred))
print('RMSE:', mean_squared_error(y_test, tree_pred) ** 0.5)
print('R2 Score:', r2_score(y_test, tree_pred))

In [ ]:
# Random Forest Regressor
forest_model = RandomForestRegressor(random_state=42)
forest_model.fit(X_train_scaled, y_train)
forest_pred = forest_model.predict(X_test_scaled)

print('Random Forest Results')
print('MAE:', mean_absolute_error(y_test, forest_pred))
print('MSE:', mean_squared_error(y_test, forest_pred))
print('RMSE:', mean_squared_error(y_test, forest_pred) ** 0.5)
print('R2 Score:', r2_score(y_test, forest_pred))

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'MAE': [mean_absolute_error(y_test, linear_pred), mean_absolute_error(y_test, tree_pred), mean_absolute_error(y_test, forest_pred)],
    'MSE': [mean_squared_error(y_test, linear_pred), mean_squared_error(y_test, tree_pred), mean_squared_error(y_test, forest_pred)],
    'RMSE': [mean_squared_error(y_test, linear_pred) ** 0.5, mean_squared_error(y_test, tree_pred) ** 0.5, mean_squared_error(y_test, forest_pred) ** 0.5],
    'R2 Score': [r2_score(y_test, linear_pred), r2_score(y_test, tree_pred), r2_score(y_test, forest_pred)]
})

print('Model Comparison')
display(results)

best_model_name = results.loc[results['MAE'].idxmin(), 'Model']
print('Best Model:', best_model_name)

In [ ]:
# Prediction using one record from the test set
if best_model_name == 'Linear Regression':
    best_model = linear_model
elif best_model_name == 'Decision Tree':
    best_model = tree_model
else:
    best_model = forest_model

new_lap = X_test_scaled[0].reshape(1, -1)
predicted_lap_time = best_model.predict(new_lap)[0]

print('Lap Time Prediction')
print('Actual Lap Time:', y_test.iloc[0])
print('Predicted Lap Time:', predicted_lap_time)
print('Absolute Error:', abs(y_test.iloc[0] - predicted_lap_time))